In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
def combine_my_data_csvs(input_dir: Path, output_csv: Path, match_str: str) -> None:
    # Find CSV files starting with match_str
    files = sorted(input_dir.glob(match_str+"*.csv"))

    if not files:
        print(f"No files matching '{match_str}'*.csv' found in: {input_dir}")
        return

    dataframes = []
    # Read each CSV
    for f in files:
        try:
            df = pd.read_csv(f)
            dataframes.append(df)
        except Exception as e:
            print(f"Skipping '{f}' due to read error: {e}")

    if not dataframes:
        print("No CSVs could be read successfully.")
        return

    # Concatenate, aligning columns. ignore_index resets the index
    combined = pd.concat(dataframes, ignore_index=True, sort=False)

    # Write to CSV
    combined.to_csv(output_csv, index=False)
    print(f"Combined {len(files)} file(s) into: {output_csv}")
    print(f"Rows: {len(combined):,} | Columns: {len(combined.columns)}")


In [ ]:
## Define paths for data inputs and outputs 
data_dir = Path("./output")
match_str = "pi_parallel_out"
output_csv = data_dir / Path("scaling_study_combined.csv")

In [ ]:
combine_my_data_csvs(data_dir, output_csv, match_str)

In [ ]:
df = pd.read_csv(output_csv)
print(df.head())

df['elapsed sec'] = df['elapsed']

## compute the average time for single core jobs 
avg_elapsed_sec_ncpus1 = df[df['ncpus'] == 1]['elapsed sec'].mean()
print('single core time : ', avg_elapsed_sec_ncpus1)

## compute the multicore speedup 
df['speedup'] = avg_elapsed_sec_ncpus1/df['elapsed sec']

print(df.head())

In [ ]:
## plot elapsed time vs. ncpus
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=df, x='ncpus', y='elapsed sec', color="darkblue", ax=ax)

In [ ]:
## plot multicore speedup vs. ncpus
## with line for what ideal parallel 
## scaling would look like

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(12, 5))

sns.scatterplot(data=df, x='ncpus', y='speedup', color="darkblue", ax=ax)

unique_x = np.unique(df['ncpus'])
plt.xticks(unique_x[1:])
plt.yticks(unique_x[1:])

xy = np.linspace(df['ncpus'].min(), df['ncpus'].max()/2, 100)
plt.plot(xy, xy, color='k', linestyle='-', label='x=y')

In [ ]:
## Run `seff` on jobids in the data files to get CPU 
## efficiency information 